# Figuring Out Test Simulation Parameters for elfe3D_GPR

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Essential Functions

In [2]:
def wavelength_ranges(f_list, wavelength, frequency, eps_r=1.0):
    """
    Function to calculate how to split the wavelength range using the frequency such that the consecutive wavelengths are half of the previous one.
    """
    f_low = f_list[0]
    f_high = f_list[-1]

    f_ranges = []
    wl_ranges = []
    dx = []
    vol = []
    antenna_lengths = []
    domain_lengths = []
    receiver_offsets = []

    wl_low_f = wavelength(f_low, eps_r)
    wl_high_f = wavelength(f_high, eps_r)

    factor = wl_low_f / wl_high_f

    num_meshes = int(np.ceil(np.log2(factor)))

    for i in range(num_meshes):
        w1 = wl_low_f / 2**i        # wavelength at the start of the range: largest wavelength
        w2 = wl_low_f / 2**(i+1)    # wavelength at the end of the range: smallest wavelength

        dx_i = w2/20 # 20 points per the smallest wavelength
        vol_i = dx_i**3/(6*np.sqrt(2))
        antenna_length_i = w1/2 # half of the largest wavelength
        domain_length_i = 10*w1 # 10 times the largest wavelength
        receiver_offset_i = [1 * antenna_length_i, 2 * antenna_length_i, 4 * antenna_length_i, 8 * antenna_length_i]

        f1 = frequency(w1, eps_r)
        f2 = frequency(w2, eps_r)

        wl_ranges.append([round(w1,5), round(w2,5)])
        f_ranges.append([f"{f1:.2e}", f"{f2:.2e}"])
        dx.append(dx_i)
        vol.append(vol_i)
        antenna_lengths.append(antenna_length_i)
        domain_lengths.append(domain_length_i)
        receiver_offsets.append(receiver_offset_i)

    return num_meshes, wl_ranges, f_ranges, dx, vol, antenna_lengths, domain_lengths, receiver_offsets

In [3]:
def logspace_frequencies(f_low, f_high, num_points):
    """
    Function to calculate the frequency points in logspace.
    """
    f_list = np.logspace(np.log10(f_low), np.log10(f_high), num_points)
    return f_list

In [4]:
# Typical constant and lambdas
c = 3e8 # speed of light in vacuum, m/s
wavelength = lambda f, eps_r: float(c/(f*np.sqrt(eps_r))) # in meters
frequency = lambda wl, eps_r: float(c/(wl*np.sqrt(eps_r))) # in Hz
velocity = lambda eps_r: float(c/np.sqrt(eps_r)) # in m/s
travel_time = lambda dx, eps_r: float(dx/velocity(eps_r)) # in seconds
fresnel_zone_area = lambda wl, dx: float(np.pi *(wl*dx)) # in square meters - radius = lambda*dx

## Number of Different Meshes, Resolution, and Antenna Lengths: Based on Frequency and Wavelength

- My idea: Divide the typical GPR freqeuncy range [10 MHz, 1 GHz] into bands depending on their wavelength.
- Each band will have a smallest wavelength limit (sw) and a largest wavelength limit (lw) such that the higher wavelength is half the lower wavelength.
- I create a mesh that is discretised by max edge length of sw/20.
- Any lower and there is a chance of numerical dispersion, any higher and I think the computational cost will be too much.
- From calculations, I will have 7 meshes, with the first mesh going from 10 MHz to 20 MHz, and the last mesh going from 640 MHz to 1.28 GHz.
- Similarly, source antenna length is preliminarily decided to be half of the largest air wavelength, and the domain length is 10 times the largest air wavelength. This means with antenna centered in acqisition plane, each side will have 5 air wavelengths worth of extents.
- CO receiver offset = length of the antenna
- Introduce 3 more points: 2x, 4x, 8x length of the antenna - check for the domain length is reached - 10x length of antenna

In [5]:
f_list = [10e6, 1e9] # 10 MHz to 1 GHz
eps_r = 1

num_meshes, wl_ranges_air, f_ranges, dx, vol, antenna_lengths, domain_lengths, receiver_offsets = wavelength_ranges(f_list, wavelength, frequency, eps_r)

# Convert to pandas dataframe
lambda_f_sets = pd.DataFrame({
    'Mesh': range(1, num_meshes + 1),
    'Wavelength Range (in Air) (m)': wl_ranges_air,
    'Frequency Range (Hz)': f_ranges,
    'Spatial Resolution (in Air) (m)': dx,
    'Volume Constraint (m^3)': vol,
    'Antenna Length (m)': antenna_lengths,
    'Domain Length (m)': domain_lengths,
    'Receiver Offsets (m)': receiver_offsets
})

In [6]:
lambda_f_sets

,Mesh,Wavelength Range (in Air) (m),Frequency Range (Hz),Spatial Resolution (in Air) (m),Volume Constraint (m^3),Antenna Length (m),Domain Length (m),Receiver Offsets (m)
0,1,"[30.0, 15.0]","[1.00e+07, 2.00e+07]",0.750000,4.971845e-02,15.000000,300.0000,"[15.0, 30.0, 60.0, 120.0]"
1,2,"[15.0, 7.5]","[2.00e+07, 4.00e+07]",0.375000,6.214806e-03,7.500000,150.0000,"[7.5, 15.0, 30.0, 60.0]"
2,3,"[7.5, 3.75]","[4.00e+07, 8.00e+07]",0.187500,7.768507e-04,3.750000,75.0000,"[3.75, 7.5, 15.0, 30.0]"
3,4,"[3.75, 1.875]","[8.00e+07, 1.60e+08]",0.093750,9.710634e-05,1.875000,37.5000,"[1.875, 3.75, 7.5, 15.0]"
4,5,"[1.875, 0.9375]","[1.60e+08, 3.20e+08]",0.046875,1.213829e-05,0.937500,18.7500,"[0.9375, 1.875, 3.75, 7.5]"
5,6,"[0.9375, 0.46875]","[3.20e+08, 6.40e+08]",0.023438,1.517287e-06,0.468750,9.3750,"[0.46875, 0.9375, 1.875, 3.75]"
6,7,"[0.46875, 0.23438]","[6.40e+08, 1.28e+09]",0.011719,1.896608e-07,0.234375,4.6875,"[0.234375, 0.46875, 0.9375, 1.875]"


## Different Wavelengths and Edge Lengths for Different Relative Permittivities

- Here, I take a variety of different relative permittivites that are commonly observed in GPR applications, and evaluate their wavelength ranges.

In [7]:
eps_r_range = [3, 4, 6, 9, 12, 15, 23]

all_f_dx = []

all_f_dx_df = {}
for eps_r in eps_r_range:
    num_meshes, wl_ranges, f_ranges, dx, vol, _, _, _ = wavelength_ranges(f_list, wavelength, frequency, eps_r)
    
    data = []
    for i in range(num_meshes):
        data.append({
            'Mesh': i + 1,
            f'Wavelength Range for ε_r = {eps_r} (m)': wl_ranges[i],
            'Frequency Range (Hz)': f_ranges[i],
            f'Spatial Resolution for ε_r = {eps_r} (m)': dx[i],
            f'Volume Constraint (m^3)': vol[i]
        })
    
    all_f_dx_df[eps_r] = pd.DataFrame(data)


In [8]:
all_f_dx_df[eps_r_range[0]]

,Mesh,Wavelength Range for ε_r = 3 (m),Frequency Range (Hz),Spatial Resolution for ε_r = 3 (m),Volume Constraint (m^3)
0,1,"[17.32051, 8.66025]","[1.00e+07, 2.00e+07]",0.433013,9.568319e-03
1,2,"[8.66025, 4.33013]","[2.00e+07, 4.00e+07]",0.216506,1.196040e-03
2,3,"[4.33013, 2.16506]","[4.00e+07, 8.00e+07]",0.108253,1.495050e-04
3,4,"[2.16506, 1.08253]","[8.00e+07, 1.60e+08]",0.054127,1.868812e-05
4,5,"[1.08253, 0.54127]","[1.60e+08, 3.20e+08]",0.027063,2.336015e-06
5,6,"[0.54127, 0.27063]","[3.20e+08, 6.40e+08]",0.013532,2.920019e-07
6,7,"[0.27063, 0.13532]","[6.40e+08, 1.28e+09]",0.006766,3.650024e-08


In [9]:
all_f_dx_df[eps_r_range[1]]

,Mesh,Wavelength Range for ε_r = 4 (m),Frequency Range (Hz),Spatial Resolution for ε_r = 4 (m),Volume Constraint (m^3)
0,1,"[15.0, 7.5]","[1.00e+07, 2.00e+07]",0.375000,6.214806e-03
1,2,"[7.5, 3.75]","[2.00e+07, 4.00e+07]",0.187500,7.768507e-04
2,3,"[3.75, 1.875]","[4.00e+07, 8.00e+07]",0.093750,9.710634e-05
3,4,"[1.875, 0.9375]","[8.00e+07, 1.60e+08]",0.046875,1.213829e-05
4,5,"[0.9375, 0.46875]","[1.60e+08, 3.20e+08]",0.023438,1.517287e-06
5,6,"[0.46875, 0.23438]","[3.20e+08, 6.40e+08]",0.011719,1.896608e-07
6,7,"[0.23438, 0.11719]","[6.40e+08, 1.28e+09]",0.005859,2.370760e-08


In [10]:
all_f_dx_df[eps_r_range[2]]

,Mesh,Wavelength Range for ε_r = 6 (m),Frequency Range (Hz),Spatial Resolution for ε_r = 6 (m),Volume Constraint (m^3)
0,1,"[12.24745, 6.12372]","[1.00e+07, 2.00e+07]",0.306186,3.382912e-03
1,2,"[6.12372, 3.06186]","[2.00e+07, 4.00e+07]",0.153093,4.228640e-04
2,3,"[3.06186, 1.53093]","[4.00e+07, 8.00e+07]",0.076547,5.285800e-05
3,4,"[1.53093, 0.76547]","[8.00e+07, 1.60e+08]",0.038273,6.607249e-06
4,5,"[0.76547, 0.38273]","[1.60e+08, 3.20e+08]",0.019137,8.259062e-07
5,6,"[0.38273, 0.19137]","[3.20e+08, 6.40e+08]",0.009568,1.032383e-07
6,7,"[0.19137, 0.09568]","[6.40e+08, 1.28e+09]",0.004784,1.290478e-08


In [11]:
all_f_dx_df[eps_r_range[3]]

,Mesh,Wavelength Range for ε_r = 9 (m),Frequency Range (Hz),Spatial Resolution for ε_r = 9 (m),Volume Constraint (m^3)
0,1,"[10.0, 5.0]","[1.00e+07, 2.00e+07]",0.250000,1.841424e-03
1,2,"[5.0, 2.5]","[2.00e+07, 4.00e+07]",0.125000,2.301780e-04
2,3,"[2.5, 1.25]","[4.00e+07, 8.00e+07]",0.062500,2.877225e-05
3,4,"[1.25, 0.625]","[8.00e+07, 1.60e+08]",0.031250,3.596531e-06
4,5,"[0.625, 0.3125]","[1.60e+08, 3.20e+08]",0.015625,4.495664e-07
5,6,"[0.3125, 0.15625]","[3.20e+08, 6.40e+08]",0.007812,5.619580e-08
6,7,"[0.15625, 0.07812]","[6.40e+08, 1.28e+09]",0.003906,7.024475e-09


In [12]:
all_f_dx_df[eps_r_range[4]]

,Mesh,Wavelength Range for ε_r = 12 (m),Frequency Range (Hz),Spatial Resolution for ε_r = 12 (m),Volume Constraint (m^3)
0,1,"[8.66025, 4.33013]","[1.00e+07, 2.00e+07]",0.216506,1.196040e-03
1,2,"[4.33013, 2.16506]","[2.00e+07, 4.00e+07]",0.108253,1.495050e-04
2,3,"[2.16506, 1.08253]","[4.00e+07, 8.00e+07]",0.054127,1.868812e-05
3,4,"[1.08253, 0.54127]","[8.00e+07, 1.60e+08]",0.027063,2.336015e-06
4,5,"[0.54127, 0.27063]","[1.60e+08, 3.20e+08]",0.013532,2.920019e-07
5,6,"[0.27063, 0.13532]","[3.20e+08, 6.40e+08]",0.006766,3.650024e-08
6,7,"[0.13532, 0.06766]","[6.40e+08, 1.28e+09]",0.003383,4.562530e-09


In [13]:
all_f_dx_df[eps_r_range[5]]

,Mesh,Wavelength Range for ε_r = 15 (m),Frequency Range (Hz),Spatial Resolution for ε_r = 15 (m),Volume Constraint (m^3)
0,1,"[7.74597, 3.87298]","[1.00e+07, 2.00e+07]",0.193649,8.558165e-04
1,2,"[3.87298, 1.93649]","[2.00e+07, 4.00e+07]",0.096825,1.069771e-04
2,3,"[1.93649, 0.96825]","[4.00e+07, 8.00e+07]",0.048412,1.337213e-05
3,4,"[0.96825, 0.48412]","[8.00e+07, 1.60e+08]",0.024206,1.671517e-06
4,5,"[0.48412, 0.24206]","[1.60e+08, 3.20e+08]",0.012103,2.089396e-07
5,6,"[0.24206, 0.12103]","[3.20e+08, 6.40e+08]",0.006052,2.611745e-08
6,7,"[0.12103, 0.06052]","[6.40e+08, 1.28e+09]",0.003026,3.264681e-09


In [14]:
all_f_dx_df[eps_r_range[6]]

,Mesh,Wavelength Range for ε_r = 23 (m),Frequency Range (Hz),Spatial Resolution for ε_r = 23 (m),Volume Constraint (m^3)
0,1,"[6.25543, 3.12772]","[1.00e+07, 2.00e+07]",0.156386,4.507397e-04
1,2,"[3.12772, 1.56386]","[2.00e+07, 4.00e+07]",0.078193,5.634246e-05
2,3,"[1.56386, 0.78193]","[4.00e+07, 8.00e+07]",0.039096,7.042807e-06
3,4,"[0.78193, 0.39096]","[8.00e+07, 1.60e+08]",0.019548,8.803509e-07
4,5,"[0.39096, 0.19548]","[1.60e+08, 3.20e+08]",0.009774,1.100439e-07
5,6,"[0.19548, 0.09774]","[3.20e+08, 6.40e+08]",0.004887,1.375548e-08
6,7,"[0.09774, 0.04887]","[6.40e+08, 1.28e+09]",0.002444,1.719435e-09


## Frequency Test Points for All Meshes

In [15]:
# For each mesh, calculate the logspace frequency points
f_points_mesh = []
num_f_points = 11

for i in range(num_meshes):
    f_low = float(f_ranges[i][0])
    f_high = float(f_ranges[i][1])
    f_points = logspace_frequencies(f_low, f_high, num_f_points)
    f_points_mesh.append(f_points)

# Convert to pandas dataframe
f_points_df = pd.DataFrame(f_points_mesh).T
f_points_df.columns = [f'Mesh {i+1}' for i in range(num_meshes)]

In [16]:
f_points_df

,Mesh 1,Mesh 2,Mesh 3,Mesh 4,Mesh 5,Mesh 6,Mesh 7
0,1.000000e+07,2.000000e+07,4.000000e+07,8.000000e+07,1.600000e+08,3.200000e+08,6.400000e+08
1,1.071773e+07,2.143547e+07,4.287094e+07,8.574188e+07,1.714838e+08,3.429675e+08,6.859350e+08
2,1.148698e+07,2.297397e+07,4.594793e+07,9.189587e+07,1.837917e+08,3.675835e+08,7.351669e+08
3,1.231144e+07,2.462289e+07,4.924578e+07,9.849155e+07,1.969831e+08,3.939662e+08,7.879324e+08
4,1.319508e+07,2.639016e+07,5.278032e+07,1.055606e+08,2.111213e+08,4.222425e+08,8.444851e+08
5,1.414214e+07,2.828427e+07,5.656854e+07,1.131371e+08,2.262742e+08,4.525483e+08,9.050967e+08
6,1.515717e+07,3.031433e+07,6.062866e+07,1.212573e+08,2.425147e+08,4.850293e+08,9.700586e+08
7,1.624505e+07,3.249010e+07,6.498019e+07,1.299604e+08,2.599208e+08,5.198415e+08,1.039683e+09
8,1.741101e+07,3.482202e+07,6.964405e+07,1.392881e+08,2.785762e+08,5.571524e+08,1.114305e+09
9,1.866066e+07,3.732132e+07,7.464264e+07,1.492853e+08,2.985706e+08,5.971411e+08,1.194282e+09


In [17]:
# print just the mesh 6 frequencies as a list
f_points_df['Mesh 5'].tolist()

[160000000.0000001,
 171483754.00580728,
 183791736.79952556,
 196983106.13518688,
 211121265.72366285,
 226274169.97969535,
 242514650.64166424,
 259920766.8339953,
 278576180.25476015,
 298570557.2917781,
 320000000.00000024]

# Layered Model 1: Air, and Earth divided into 2 layers (Aquifer Model)

| Layer | Material                                  | Relative Permittivity (εr) | Conductivity (σ) [S/m] | Thickness [m] |
|-------|------------------------------------------|---------------------------|-----------------------|---------------|
| 1     | Unsaturated Dry Sandy Soil              | 3–4                       | 0.001–0.01            | ~2            |
| 2     | Saturated Aquifer (Water-Saturated Sand/Gravel) | 20–25  | 0.01–0.05            | ~8+           |

In [18]:
# Apply the wavelength lambda function to the frequency ranges in the DataFrame
lambda_f_sets['Wavelength Range for ε_r = 3 (m)'] = lambda_f_sets['Frequency Range (Hz)'].apply(lambda x: [wavelength(float(x[0]), 3), wavelength(float(x[1]), 3)])
lambda_f_sets['Wavelength Range for ε_r = 20 (m)'] = lambda_f_sets['Frequency Range (Hz)'].apply(lambda x: [wavelength(float(x[0]), 20), wavelength(float(x[1]), 20)])

print(lambda_f_sets[['Frequency Range (Hz)', 'Wavelength Range for ε_r = 3 (m)', 'Wavelength Range for ε_r = 20 (m)']])

   Frequency Range (Hz)           Wavelength Range for ε_r = 3 (m)  \
0  [1.00e+07, 2.00e+07]    [17.320508075688775, 8.660254037844387]   
1  [2.00e+07, 4.00e+07]     [8.660254037844387, 4.330127018922194]   
2  [4.00e+07, 8.00e+07]     [4.330127018922194, 2.165063509461097]   
3  [8.00e+07, 1.60e+08]    [2.165063509461097, 1.0825317547305484]   
4  [1.60e+08, 3.20e+08]   [1.0825317547305484, 0.5412658773652742]   
5  [3.20e+08, 6.40e+08]   [0.5412658773652742, 0.2706329386826371]   
6  [6.40e+08, 1.28e+09]  [0.2706329386826371, 0.13531646934131855]   

            Wavelength Range for ε_r = 20 (m)  
0    [6.7082039324993685, 3.3541019662496843]  
1    [3.3541019662496843, 1.6770509831248421]  
2    [1.6770509831248421, 0.8385254915624211]  
3   [0.8385254915624211, 0.41926274578121053]  
4  [0.41926274578121053, 0.20963137289060527]  
5  [0.20963137289060527, 0.10481568644530263]  
6  [0.10481568644530263, 0.05240784322265132]  


# Layered Model 2: Air, and Earth divided into 3 layers (Stratigraphic Model)

| Layer | Material                                  | Relative Permittivity (εr) | Conductivity (σ) [S/m] | Thickness [m] |
|-------|------------------------------------------|---------------------------|-----------------------|---------------|
| 1     | Unsaturated Dry Sandy Soil              | 3–4                       | 0.001–0.01            | ~1            |
| 2     | Moist Clay/Silty Soil                   | 12–20 (~15 typical)       | 0.1–0.5               | ~1.5–2        |
| 3     | Saturated Aquifer (Water-Saturated Sand/Gravel) | 20–25  | 0.01–0.05            | ~10+          |

In [19]:
# Apply the wavelength lambda function to the frequency ranges in the DataFrame
lambda_f_sets['Wavelength Range for ε_r = 15 (m)'] = lambda_f_sets['Frequency Range (Hz)'].apply(lambda x: [wavelength(float(x[0]), 15), wavelength(float(x[1]), 15)])

print(lambda_f_sets[['Frequency Range (Hz)', 'Wavelength Range for ε_r = 15 (m)']])

   Frequency Range (Hz)           Wavelength Range for ε_r = 15 (m)
0  [1.00e+07, 2.00e+07]      [7.745966692414834, 3.872983346207417]
1  [2.00e+07, 4.00e+07]     [3.872983346207417, 1.9364916731037085]
2  [4.00e+07, 8.00e+07]    [1.9364916731037085, 0.9682458365518543]
3  [8.00e+07, 1.60e+08]    [0.9682458365518543, 0.4841229182759271]
4  [1.60e+08, 3.20e+08]   [0.4841229182759271, 0.24206145913796356]
5  [3.20e+08, 6.40e+08]  [0.24206145913796356, 0.12103072956898178]
6  [6.40e+08, 1.28e+09]  [0.12103072956898178, 0.06051536478449089]


# Layered Model 3: Air, and Earth divided into 4 layers (Archaeological Model)

| Layer | Material                     | Relative Permittivity (εr) | Conductivity (σ) [S/m] | Thickness [m] |
|-------|------------------------------|---------------------------|-----------------------|---------------|
| 1     | Topsoil (Sand)          | 4–6                       | 0.005–0.02            | ~0.5          |
| 2     | Compacted Earth              | 5–8                       | 0.01–0.03             | ~0.5–1        |
| 3     | Buried Structure (Brick/Stone) | 6–10                     | 0.001–0.01            | ~1            |
| 4     | Natural Subsoil (Clayey Sand) | 10–15                     | 0.05–0.2              | ~5+           |

In [20]:
# Apply the wavelength lambda function to the frequency ranges in the DataFrame
lambda_f_sets['Wavelength Range for ε_r = 5 (m)'] = lambda_f_sets['Frequency Range (Hz)'].apply(lambda x: [wavelength(float(x[0]), 5), wavelength(float(x[1]), 5)])
lambda_f_sets['Wavelength Range for ε_r = 6 (m)'] = lambda_f_sets['Frequency Range (Hz)'].apply(lambda x: [wavelength(float(x[0]), 6), wavelength(float(x[1]), 6)])
lambda_f_sets['Wavelength Range for ε_r = 9 (m)'] = lambda_f_sets['Frequency Range (Hz)'].apply(lambda x: [wavelength(float(x[0]), 9), wavelength(float(x[1]), 9)])
lambda_f_sets['Wavelength Range for ε_r = 12 (m)'] = lambda_f_sets['Frequency Range (Hz)'].apply(lambda x: [wavelength(float(x[0]), 12), wavelength(float(x[1]), 12)])

print(lambda_f_sets[['Frequency Range (Hz)', 'Wavelength Range for ε_r = 5 (m)', 'Wavelength Range for ε_r = 6 (m)', 'Wavelength Range for ε_r = 9 (m)', 'Wavelength Range for ε_r = 12 (m)']])

   Frequency Range (Hz)            Wavelength Range for ε_r = 5 (m)  \
0  [1.00e+07, 2.00e+07]    [13.416407864998737, 6.7082039324993685]   
1  [2.00e+07, 4.00e+07]    [6.7082039324993685, 3.3541019662496843]   
2  [4.00e+07, 8.00e+07]    [3.3541019662496843, 1.6770509831248421]   
3  [8.00e+07, 1.60e+08]    [1.6770509831248421, 0.8385254915624211]   
4  [1.60e+08, 3.20e+08]   [0.8385254915624211, 0.41926274578121053]   
5  [3.20e+08, 6.40e+08]  [0.41926274578121053, 0.20963137289060527]   
6  [6.40e+08, 1.28e+09]  [0.20963137289060527, 0.10481568644530263]   

             Wavelength Range for ε_r = 6 (m)  \
0      [12.24744871391589, 6.123724356957945]   
1     [6.123724356957945, 3.0618621784789726]   
2    [3.0618621784789726, 1.5309310892394863]   
3    [1.5309310892394863, 0.7654655446197431]   
4   [0.7654655446197431, 0.38273277230987157]   
5  [0.38273277230987157, 0.19136638615493579]   
6  [0.19136638615493579, 0.09568319307746789]   

  Wavelength Range for ε_r = 9 (m)    

## Volume vs Edge Discretization

- For Volume: Use the formula: V = (lambda/20)^3/(6*sqrt(2)). Check how to replace lambda/20 such that the tetrahedra are always having edge less than lambda/20.
    - I do not implement this, since it is not realistically possible to constrain this, in case there are very skewed tetrahedra.
- For Edge: Check how to make the upper layer have a "transition layer" such that its discretisation goes from its own lambda/20 to match the lambda/20 exactly of the layer below.
    - If a layer is thicker than one wavelength of its own medium, I assign thickness equal to one wavelength of the medium as a transition zone.
    - If not, the whole layer will have varying mesh size.
    - I implement this.

## Truncated Domain Calculation
- Illumination Zone
- First Fresnel Zone
- Enough Domain to look at possible reflections from the bottom: Travel time based calculation? Frequency domain - eval travel time from phase: 180, then use it to get distance.

## Final Layered Models:

1. Domain Length
2. Interface Heights
3. Transition Zone Heights
4. Material Properties of Layers
5. Source Location
6. Source Orientation
7. Source Frequency Range
8. Inline Receiver Locations
9. End-fire Receiver Locations
10. Fresnel Zone Area for final simulations with PML?

# Layered Model 1: Air, and Earth divided into 2 layers (Aquifer Model)

- Thickness of Air chosen by rounding up 5 wavelengths from target frequency.

1. Domain Length (Horizontal Extents): [18.7500], [9.3750], [4.6875]
2. Thicknesses: [10, 2, 8]
3. Transition Zone Thickness (~one wavelength height): [1.875, 1.08253], [0.9375, 0.54127], [0.46875, 0.27063]
5. Source Location: Origin
6. Source Orientation: x directed dipole
7. Source Frequency Range: [160 MHz, 320 MHz], [320 MHz, 640 MHz], [640 MHz, 1.28 GHz] 
8. Inline Receiver Locations: Check the lists from mesh 5 for x coordinates, y, z = 0
9. End-fire Receiver Locations: Check the lists from mesh 5 for y coordinates, x, z = 0

Receivers Lists:
- [0.9375, 1.875, 3.75, 7.5]
- [0.46875, 0.9375, 1.875, 3.75]
- [0.234375, 0.46875, 0.9375, 1.875]

| Layer | Material                                  | εr | σ [S/m] | Thickness [m] |
|-------|------------------------------------------|---------------------------|-----------------------|---------------|
| 1     | Air | 1 | 1e-14 | 10 |
| 2     | Unsaturated Dry Sandy Soil              | 3                       | 0.005            | 2            |
| 3     | Saturated Aquifer (Water-Saturated Sand/Gravel) | 23  | 0.025            | 8           |




# Layered Model 2: Air, and Earth divided into 3 layers (Stratigraphic Model)

1. Domain Length (Horizontal Extents): [9.375], [4.6875]
2. Thicknesses: [5, 1, 2, 10]
3. Transition Zone Thickness (~ one wavelength height): [0.9375, 0.54127, 0.24206], [0.46875, 0.27063, 0.12103]
5. Source Location: Origin
6. Source Orientation: x directed dipole
7. Source Frequency Range: [320 MHz, 640 MHz], [640 MHz, 1.28 GHz] 
8. Inline Receiver Locations: Check the lists from mesh 6 for x coordinates, y, z = 0
9. End-fire Receiver Locations: Check the lists from mesh 6 for y coordinates, x, z = 0

Receivers' List:
- [0.46875, 0.9375, 1.875, 3.75]
- [0.234375, 0.46875, 0.9375, 1.875]

| Layer | Material                                  |  εr | σ [S/m] | Thickness [m] |
|-------|------------------------------------------|---------------------------|-----------------------|---------------|
| 1     | Air | 1 | 1e-14 | 5 |
| 2     | Unsaturated Dry Sandy Soil              | 3                       | 0.005            | 1            |
| 3     | Moist Clay/Silty Soil                   | 15       | 0.25               | 2        |
| 4     | Saturated Aquifer (Water-Saturated Sand/Gravel) | 23  | 0.025            | 10          |

# Layered Model 3: Air, and Earth divided into 4 layers (Archaeological Model)

1. Domain Length (Horizontal Extents): [9.375], [4.6875]
2. Thicknesses: [5, 0.5, 0.75, 1, 5]
3. Transition Zone Thickness: [0.9375, -, 0.38273, 0.3125], [0.46875, 0.23438, 0.19137, 0.15625]
5. Source Location: Origin
6. Source Orientation: x directed dipole
7. Source Frequency Range: [320 MHz, 640 MHz], [640 MHz, 1.28 GHz] 
8. Inline Receiver Locations: Check the lists from mesh 6 for x coordinates, y, z = 0
9. End-fire Receiver Locations: Check the lists from mesh 6 for y coordinates, x, z = 0

Receivers' List:
- [0.46875, 0.9375, 1.875, 3.75]
- [0.234375, 0.46875, 0.9375, 1.875]

| Layer | Material                     | εr | σ [S/m] | Thickness [m] |
|-------|------------------------------|---------------------------|-----------------------|---------------|
| 1     | Air | 1 | 1e-14 | 10 |
| 2     | Topsoil (Loam/Sand)          | 4                       | 0.01            | 0.5          |
| 3     | Compacted Earth              | 6                       | 0.02             | 0.75        |
| 4     | Buried Structure (Brick/Stone) | 9                     | 0.005            | 1            |
| 5    | Natural Subsoil (Clayey Sand) | 12                     | 0.1              | 5           |